In [ ]:
# S4: Window Functions
# Window functions perform calculations across a set of rows that are related to the current row
# GROUP BY collapses rows
# WINDOW functions allow rows to keep their identity while also aggregating information about their grouping

# Without window function (GROUP BY):     With window function:
# supplier   total_spend                  po_id  supplier  amount  supplier_total
# Acme       819898                       PO-001 Acme      32055   819898
# GlobalCo   752095                       PO-002 Acme      22382   819898
# FastParts  995412                       PO-003 GlobalCo  47315   752095
#                                         PO-004 FastParts 23470   995412


# WINDOW FUNCTION documentation

# function_name() OVER (
#     PARTITION BY column    -- define the groups (like GROUP BY)
#     ORDER BY column        -- define the order within each group
#     ROWS/RANGE BETWEEN ... -- optional: define the window frame
# )

# PARTITION BY — divides rows into groups. Like GROUP BY but rows aren't collapsed

# ORDER BY — defines the order within each partition — required for ranking and running totals

# OVER() — what makes it a window function. Without OVER() it's a regular aggregate



In [ ]:
# Six Important Window Functions to Practice

# ROW_NUMBER() — unique sequential number per partition -- no ties allowed -- assigns a unique # to each row

# RANK() — ranking with gaps for ties -- tied values get the same rank -- next rank is skipped [2,2,2,4] <-- no 3

# DENSE_RANK() — ranking without gaps for ties -- applied consecutive tiers no skips [2,2,2,3,4,4,5]

# LAG() — access a previous row's value -- returns the value from N rows AFTER the current row

# LEAD() — access a next row's value -- returns the value from N rows AFTER the current row

# SUM() OVER() — running total -- cumulative sum within a partition based on a columns order

# High Level Summary

# GROUP BY collapses rows -- SC example would be collapsing all rows so there is 1 per Supplier or SKU or Warehouse
# PARTITION BY keeps all rows -- SC example would add a total alongside each row/instance in your data

In [1]:
import pandas as pd
import numpy as np
%load_ext sql
%sql duckdb:///:memory:

np.random.seed(22)
suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMg", "EastCoast"]
categories = ["Electronics", "Hardware", "Consumables", "Apparel"]
warehouses = ["East", "West", "Central"]

rows = []
for i in range(1, 201):
    rows.append({
        "po_id":          f"PO-{str(i).zfill(4)}",
        "supplier":       np.random.choice(suppliers),
        "category":       np.random.choice(categories),
        "warehouse":      np.random.choice(warehouses),
        "order_date":     pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.randint(0, 365))),
        "amount":         round(np.random.uniform(500, 50000), 2),
        "units":          np.random.randint(10, 500),
        "lead_time_days": np.random.randint(3, 45),
        "on_time":        np.random.choice([True, False], p=[0.8, 0.2])
    })

po = pd.DataFrame(rows)
%sql --persist po
print(f"po: {po.shape}")

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

Success! Persisted po to the database.

po: (200, 9)


In [3]:
po.head(5)

,po_id,supplier,category,warehouse,order_date,amount,units,lead_time_days,on_time
0,PO-0001,EastCoast,Electronics,East,2024-12-22,32055.22,94,11,True
1,PO-0002,FastParts,Consumables,East,2024-02-15,23470.66,103,42,True
2,PO-0003,Acme,Electronics,West,2024-01-28,22382.64,39,21,True
3,PO-0004,GlobalCo,Apparel,Central,2024-01-08,47315.60,33,26,True
4,PO-0005,EastCoast,Apparel,East,2024-12-25,37347.63,123,4,False


In [6]:
%%sql

-- S4.1 ROW_NUMBER()
-- Business question: what is the highest value PO per supplier?
-- step 1: use ROW_NUMBER() to rank POs within each supplier by amount descending
-- step 2: wrap in a subquery and filter where row_num = 1
-- show: supplier, po_id, amount, row_num

select
    supplier,
    po_id,
    amount,
    row_num
from (
select
    supplier,
    po_id,
    amount,
    ROW_NUMBER() OVER(
        PARTITION BY supplier -- restarts the ranking per unique supplier
        ORDER BY amount DESC
    ) AS row_num
from
    po
)
where
    row_num = 1 -- keeps only the top row per supplier
order by
    amount desc -- ensures highest ranking is the first row


Running query in 'duckdb:///:memory:'

supplier,po_id,amount,row_num
EastCoast,PO-0155,49202.14,1
Acme,PO-0083,49123.34,1
PrimeMg,PO-0046,47855.66,1
GlobalCo,PO-0045,47711.66,1
FastParts,PO-0050,47326.18,1
